In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🎯 Lead Generation Crew

## What We're Building

An outbound sales lead generation pipeline:

| Agent | Role |
|-------|------|
| **ICP Finder** | Define the Ideal Customer Profile from product description |
| **Company Researcher** | Find and profile target companies matching the ICP |
| **Contact Writer** | Identify key decision-makers at each company |
| **Email Personalization Agent** | Write personalized outreach emails |

## Key CrewAI Concept: Task Dependencies with `context`

The email personalization task explicitly depends on outputs from the
company researcher AND contact writer tasks via the `context` parameter.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Product Description

In [ ]:
product_info = """
Product: DataSync Pro
Category: B2B SaaS — Data Integration Platform
What it does: Connects disparate data sources (Salesforce, HubSpot, Snowflake,
PostgreSQL, Google Sheets) and syncs them in real-time with no-code connectors.
Key differentiator: Handles schema drift automatically; 10-minute setup vs.
competitors' multi-day onboarding.
Pricing: $499/mo (Starter), $1,499/mo (Growth), Custom (Enterprise)
Best for: Mid-market companies (50-500 employees) with messy data spread
across 5+ tools who are tired of broken integrations and manual CSV exports.
"""

print("Product loaded: DataSync Pro")

## Step 3 — Create Agents

In [ ]:
icp_finder = Agent(
    role="Ideal Customer Profile Strategist",
    goal="Define a precise ICP based on the product's value proposition",
    backstory=(
        "You are a growth marketing strategist who has built ICP frameworks for "
        "50+ B2B SaaS companies. You think in terms of firmographics (industry, size, "
        "revenue), technographics (tech stack), and psychographics (pain points, triggers)."
    ),
    llm=llm,
    verbose=True,
)

company_researcher = Agent(
    role="Target Account Researcher",
    goal="Identify and profile companies that match the ICP",
    backstory=(
        "You are an SDR team lead who has researched thousands of accounts. "
        "You know how to find relevant details about companies: their tech stack, "
        "recent funding, team size, and likely pain points that match our product."
    ),
    llm=llm,
    verbose=True,
)

contact_writer = Agent(
    role="Contact Intelligence Specialist",
    goal="Identify the right decision-makers and champions at target companies",
    backstory=(
        "You are an expert at organizational mapping. You know that for data integration "
        "tools, the buyer is usually the VP of Engineering or Head of Data, the champion "
        "is often a data engineer, and the economic buyer is the CTO or VP of Ops."
    ),
    llm=llm,
    verbose=True,
)

email_agent = Agent(
    role="Outreach Email Personalization Expert",
    goal="Write highly personalized cold outreach emails that get responses",
    backstory=(
        "You are a top-performing SDR who consistently gets 15%+ reply rates on cold email. "
        "Your secret: deep personalization based on the prospect's company, role, and likely "
        "pain points. You never send generic templates. You write short, specific emails."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks with Dependencies

In [ ]:
icp_task = Task(
    description=f"""Define the Ideal Customer Profile for this product:

{product_info}

Create a detailed ICP including:
1. Firmographics: industry, company size, revenue range, geography
2. Technographics: tools they likely use, tech maturity level
3. Pain points: specific problems they face that our product solves
4. Buying triggers: events that would make them search for a solution
5. Anti-ICP: who should we NOT target (and why)""",
    expected_output="A detailed Ideal Customer Profile document.",
    agent=icp_finder,
)

company_task = Task(
    description="""Based on the ICP defined above, generate 5 fictional but realistic
target company profiles. For each company include:
1. Company name, industry, headquarters, employee count, revenue estimate
2. Their likely tech stack (CRM, data warehouse, analytics tools)
3. Recent news or events (funding, hiring, product launch) that create urgency
4. Specific pain points relevant to our product
5. Score each company 1-10 on ICP fit""",
    expected_output="5 detailed target company profiles with ICP fit scores.",
    agent=company_researcher,
)

contact_task = Task(
    description="""For each of the 5 target companies profiled above, identify:
1. The likely DECISION MAKER (title, why they care)
2. The likely CHAMPION (title, their day-to-day pain)
3. The ECONOMIC BUYER (title, what metric they care about)
4. Recommended first point of contact and why
5. Personalization hooks (recent LinkedIn activity, shared connections, etc.)""",
    expected_output="Contact mapping for 5 companies with personalization hooks.",
    agent=contact_writer,
)

# This task explicitly depends on outputs from company_task and contact_task
email_task = Task(
    description="""Write personalized cold outreach emails for each of the 5 target companies.

For each company, write:
1. Subject line (under 50 chars, personalized, no spam words)
2. Email body (under 150 words, 3 paragraphs max)
   - Paragraph 1: Personalized hook based on their company/role
   - Paragraph 2: How DataSync Pro solves their specific pain
   - Paragraph 3: Low-friction CTA (not "book a demo" — something easier)
3. Follow-up email (sent 3 days later if no reply, even shorter)

Rules: No buzzwords, no "I hope this finds you well", no attachments.""",
    expected_output="5 personalized cold emails with follow-ups.",
    agent=email_agent,
    context=[company_task, contact_task],  # Explicit dependencies!
)

print("4 tasks created (email task depends on company + contact tasks)")

## Step 5 — Run the Crew

In [ ]:
lead_crew = Crew(
    agents=[icp_finder, company_researcher, contact_writer, email_agent],
    tasks=[icp_task, company_task, contact_task, email_task],
    process=Process.sequential,
    verbose=True,
)

print("Lead gen crew assembled! Starting pipeline...")
result = lead_crew.kickoff()
print("\n✅ Lead generation complete!")

In [ ]:
print("📧 PERSONALIZED OUTREACH EMAILS:")
print("=" * 60)
print(result.raw)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Task context** | `context=[task_a, task_b]` — explicit dependencies |
| **Role specialization** | Each agent is an expert in their narrow domain |
| **ICP framework** | Firmographics + technographics + psychographics |
| **Personalization** | Emails tailored to each company's specific situation |

## 🔧 Extensions

- **CRM integration**: Push generated leads to HubSpot/Salesforce
- **LinkedIn scraping**: Use tools to find real company data
- **A/B testing**: Generate multiple email variants per contact
- **Lead scoring**: Add a scoring agent to prioritize outreach order